<a href="https://colab.research.google.com/github/hamzafarooq/multi-agent-course/blob/main/modules/Module_3_Production_Agentic_RAG_AI_Systems/002.%20Semantic%20Caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**If you use our code, please cite:**

@misc{2024<br>
  title = {Semantic Cache from Scratch},<br>
  author = {Hamza Farooq, Darshil Modi, Kanwal Mehreen, Nazila Shafiei},<br>
  keywords = {Semantic Cache},<br>
  year = {2024},<br>
  copyright = {APACHE 2.0 license}<br>
}

## Semantic Cache

Semantic caching accelerates retrieval-augmented workflows by storing and reusing previous embedding-based lookups instead of issuing fresh queries every time. In this notebook, we'll build a lightweight semantic cache from scratch using:

- **Nomic text embeddings** (`nomic-ai/nomic-embed-text-v1.5`) to convert documents and queries into dense vectors  
- **FAISS** (Facebook AI Similarity Search) to index and quickly search those vectors  
- **SerpApi** (live Google search) as the answer backend that a cache miss falls through to  

Rather than re-computing embeddings and retrieval for every query, our cache lets us:

1. **Embed** each new query and check if it's already "covered" by a cached result  
2. **Fall back** to a full RAG retrieval (and store the new result) only when necessary  
3. **Skip the cache entirely** for time-sensitive questions that need a fresh answer  
4. **Call** the search backend only on a miss — and never for time-sensitive questions  

This approach reduces redundant compute, lowers end-to-end latency, and makes RAG pipelines more efficient—especially when query patterns exhibit repetition or high similarity. We'll walk through:

1. Loading the Nomic embed model with `trust_remote_code=True`  
2. Building a FAISS index for fast L2 nearest-neighbor lookup  
3. Implementing the core cache hit/miss logic with a time-sensitivity filter  
4. Falling back to SerpApi on a cache miss, and storing what comes back  
5. Measuring performance gains against a "no-cache" baseline  

By the end, you'll have a reusable semantic cache scaffold that you can plug into any RAG or search-over-embeddings pipeline. Let's get started!

## Semantic Caching — System Workflow

The diagram below maps every decision point in the `SemanticCaching.ask()` pipeline, from receiving a query to returning a response.

```
                         ┌──────────────────────┐
                         │      User Query       │
                         └──────────┬───────────┘
                                    │
                                    ▼
                  ┌─────────────────────────────────────┐
                  │       Time-Sensitivity Check         │
                  │        is_time_sensitive()            │
                  │                                      │
                  │  Scans for keywords such as:         │
                  │  "now", "today", "current",          │
                  │  "latest", "outage", "live",         │
                  │  "breaking", "this week", etc.       │
                  └────────────────┬────────────────────┘
                                   │
              ┌────────────────────┴────────────────────┐
         YES  │                                         │  NO
              ▼                                         ▼
 ┌────────────────────────┐             ┌──────────────────────────────┐
 │    SerpApi (live)       │             │        Embed Query            │
 │   (Live Google Search)  │             │   nomic-ai/nomic-embed-       │
 │                         │             │   text-v1.5  →  768-dim vec  │
 │  Real-time answer for   │             │   get_text_embeddings()       │
 │  time-bound questions   │             └──────────────┬───────────────┘
 │                         │                            │
 │  ✗ Result NOT stored    │                            ▼
 │    in FAISS or JSON     │           ┌────────────────────────────────┐
 └───────────┬─────────────┘           │      FAISS Index Search        │
             │                         │   IndexFlatL2 (L2 / Euclidean) │
             │                         │                                │
             │                         │  threshold = 0.2               │
             │                         │  (lower = stricter matching)   │
             │                         └──────────┬─────────────────────┘
             │                                    │
             │                   ┌────────────────┴────────────────┐
             │              HIT  │  distance ≤ 0.2                 │  MISS  distance > 0.2
             │                   ▼                                 ▼
             │     ┌──────────────────────────┐   ┌───────────────────────────────┐
             │     │      Cache Hit  ⚡         │   │     SerpApi (cache miss)      │
             │     │                           │   │     make_prediction()         │
             │     │  Fetch stored answer      │   │                               │
             │     │  from JSON file by        │   │  Sends query to hosted RAG    │
             │     │  FAISS row index          │   │  API over the AWS Guidebook   │
             │     │                           │   │  corpus — chunks retrieved,   │
             │     │  ~0.1 – 0.2 s latency     │   │  LLM generates answer         │
             │     └──────────────┬────────────┘   │                               │
             │                    │                │  ~6 – 8 s latency             │
             │                    │                └──────────────┬────────────────┘
             │                    │                               │
             │                    │                               ▼
             │                    │              ┌────────────────────────────────┐
             │                    │              │      Store in Cache             │
             │                    │              │                                │
             │                    │              │  FAISS  → add new embedding    │
             │                    │              │  JSON   → append question,     │
             │                    │              │           embedding, answer,   │
             │                    │              │           response_text         │
             │                    │              └──────────────┬─────────────────┘
             │                    │                             │
             └────────────────────┴────────────────┬───────────┘
                                                   ▼
                                    ┌──────────────────────────┐
                                    │      Final Response      │
                                    │        to User           │
                                    └──────────────────────────┘
```

---

### What Lives in the Cache?

```
  cache.json  (persisted to disk across restarts)
  ┌──────────────────────────────────────────────────────────────┐
  │ {                                                             │
  │   "questions"    : ["What is S3?", "How does Lambda work?"]  │
  │   "embeddings"   : [[0.12, -0.43, … ], [0.08, 0.71, … ]]    │  ← 768-dim Nomic vectors
  │   "answers"      : [{ raw backend result }, … ]                │
  │   "response_text": ["An S3 bucket is …", "Lambda is …"]      │
  │ }                                                             │
  └──────────────────────────────────────────────────────────────┘

  FAISS IndexFlatL2  (in-memory, rebuilt from JSON on load)
  ┌──────────────────────────────────────────────────────────────┐
  │  embedding[0]  →  row 0  →  cache["response_text"][0]        │
  │  embedding[1]  →  row 1  →  cache["response_text"][1]        │
  │  …                                                           │
  └──────────────────────────────────────────────────────────────┘
```

---

### Routing Decision Summary

| Condition | Backend | Result cached? | Typical latency |
|---|---|---|---|
| Temporal keyword detected | **SerpApi** (live Google) | ✗ Never | ~0.1 – 1.5 s |
| Stable query — FAISS hit (dist ≤ 0.2) | **FAISS + JSON store** | ✓ Already stored | ~0.1 – 0.2 s |
| Stable query — FAISS miss (dist > 0.2) | **SerpApi** | ✓ Stored after call | ~2 – 4 s |

> **Key insight:** The cache only stores answers for *stable* questions (AWS concepts, service descriptions, architectural comparisons). Time-sensitive questions — outages, live pricing, breaking news — bypass the cache entirely and always hit the live search API, preventing stale answers from ever being served.

## Setup and Dependencies

In [1]:
# Install the necessary libraries
!pip install -U faiss-cpu transformers==4.48.0 sentence-transformers==3.4.1 einops==0.8.1 python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 86.1 MB/s eta 0:00:00
  Attempting uninstall: einops
    Found existing installation: einops 0.8.2
    Uninstalling einops-0.8.2:
      Successfully uninstalled einops-0.8.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling t

In [2]:
# Import the necessary libraries

# SentenceTransformers wrapper around transformer models for text embeddings
# NOTE: this import MUST come before faiss. Both ship their own OpenMP runtime, and on
# macOS loading a SentenceTransformer after faiss aborts the process with no traceback —
# the kernel simply dies. Import order is the fix.
from sentence_transformers import SentenceTransformer  # Loads Nomic/embed or other SBERT-style models

# FAISS for efficient similarity search over vector embeddings
import faiss  # Builds and queries approximate nearest neighbor indices

# Lightweight SQL database for caching metadata, query logs, or evaluation results
import sqlite3  # Persistence layer for storing cache entries or metrics

# PyTorch backend required by SentenceTransformer and optional model fine-tuning
import torch  # Tensor operations, GPU acceleration, and model inference support

# Transformers library components for causal LLM-based answer generation
from transformers import AutoModelForCausalLM, AutoTokenizer
#   - AutoModelForCausalLM: Load pretrained language models (e.g., GPT variants)
#   - AutoTokenizer: Tokenize text input/output for the LLM

# Core numerical library for array and matrix operations on embeddings
import numpy as np  # Handles vector math, concatenation, and statistical computations

# Pretty-printing complex Python objects during development/debugging
from pprint import pprint  # Nicely formats nested dicts or lists when exploring outputs

# Define the Retrieval Function

One backend, two cache policies. Every answer in this notebook comes from **SerpApi**
(structured Google results). What changes is whether the answer is allowed to be reused:

| Question type | Backend | Cached? |
|---|---|---|
| Stable / conceptual | **SerpApi** | ✅ Yes — a hit skips the call entirely |
| Time-sensitive / live data | **SerpApi** | ❌ Never — always re-fetched |

That's the whole lesson of a semantic cache: the expensive call is the same either way,
and the only question is whether you're *allowed* to skip it. A cache in front of a
document RAG pipeline behaves identically — you'll build exactly that in
`003. Agentic Router_semantic_caching_rbac.ipynb`, where the backend is a Qdrant
retrieval + grounded generation instead of a web search.

---

## SerpApi — Live Internet Search

[SerpApi](https://serpapi.com) provides structured Google search results via a REST API.

**API details:**

| Property | Value |
|---|---|
| Endpoint | `GET https://serpapi.com/search.json` |
| Auth | `?api_key=<your_key>` query param |
| Key params | `q=<query>`, `engine=google`, `num=5` |
| Response | `organic_results[].snippet`, `answer_box` |

Sign up at [serpapi.com](https://serpapi.com) for a free key (100 searches/month).

In [3]:
import os
import requests  # HTTP client for REST API calls

# ── Credential loading — works on Colab and locally ──────────────────────────
# On Colab:  store the key in the Secrets panel (🔑 left sidebar) as SERP_API_KEY
# Locally:   it is read from .env  (SERP_API_KEY, SERPAPI_KEY or serp_api_key all work)

try:
    from google.colab import userdata
    serp_api_key = userdata.get("SERP_API_KEY")
    print("Running on Colab — credentials loaded from Secrets.")
except ImportError:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())   # walks up the directory tree to find .env
    # accept any of the common spellings
    serp_api_key = os.getenv("serp_api_key") or os.getenv("SERP_API_KEY") or os.getenv("SERPAPI_KEY")
    print("Running locally — credentials loaded from .env file.")

print(f"SerpApi key loaded: {'✅' if serp_api_key else '❌ MISSING'}")


# ── SerpApi: the answer backend ───────────────────────────────────────────────

def search_web(query: str) -> str:
    """
    Answer a question from live Google results via SerpApi.

    This is the notebook's only answer backend. It is called on a cache MISS for a
    stable question (and the result is stored), and on every time-sensitive question
    (where the result is deliberately NOT stored). Same call, different cache policy —
    that distinction is the point of the notebook.

    Args:
        query (str): The time-sensitive question to search for.

    Returns:
        str: A formatted string combining the answer box (if present) and
             top organic result snippets.
    """
    if not serp_api_key:
        raise RuntimeError("Missing SERP_API_KEY — add it to Colab Secrets or .env")

    print("SerpApi: fetching live search results 🌐 ...")
    params = {
        "q": query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        response = requests.get("https://serpapi.com/search.json", params=params)
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    except requests.exceptions.RequestException as e:
        return f"SerpApi request error: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

Running on Colab — credentials loaded from Secrets.
Traversaal Pro key loaded: ✅
SerpApi key loaded:        ✅


In [9]:
# # Quick raw-curl check of the SerpApi endpoint.
# # Uses the variable loaded above — no hard-coded key.
# !curl -s "https://serpapi.com/search.json?engine=google&num=3&q=what+is+an+s3+bucket&api_key={serp_api_key}" | head -c 600

In [5]:
# Test the backend directly — a stable question, the kind that is worth caching
result = search_web("What is an S3 bucket in AWS?")
print(result[:800])

Traversaal Pro: request successful.
Generated answer:
Unable to generate a response right now due to an LLM provider connection issue.

Top source reference:
  Score: 0.674
  File:  Amazon Simple Storage Service - User Guide.pdf
  Chunk: Amazon Simple Storage Service
User Guide
Table buckets – Recommended for storing tabular data, such as daily purchase transactions, 
streaming sensor data, or ad impressions. Tabular data represents d...


In [6]:
# The same backend on a time-sensitive question — this one must never be cached
live_answer = search_web("Are there any AWS outages right now?")
print(live_answer[:800])

SerpApi: fetching live search results 🌐 ...
[1] Service health - Sep 15, 2026 | AWS Health Dashboard | Global
    Mar 03 8:14 AM PST We are providing an update on the ongoing service disruptions affecting the AWS Middle East (UAE) Region (ME-CENTRAL-1). We continue to make ...
    Source: https://health.aws.amazon.com/

[2] AWS live status. Problems and outages for Amazon Web ...
    Amazon Web Services User reports show no current problems with Amazon Web Services
    Source: https://downdetector.com/status/aws-amazon-web-services/

[3] Service health - Sep 15, 2026 | AWS Health Dashboard | Global
    At this time, power has not yet been restored to the affected AZ. For now, we recommend continuing to retry any failed API requests.
    Source: https://health.aws.amazon.com/health/status?eventID=arn:aws:health:us-east-1::event/MULTIPLE_SERVICES/AWS_MULTIPLE_SERVICES_OPERATIONAL_ISSUE/AWS_MULTIPLE_SERVICES_OPERATIONAL_ISSUE_BA540_514A652BE1A

[4] r/aws
    The majority of my contacts th

### Define SemanticCaching Class

In this cell we define `SemanticCaching`—a lightweight cache with dual-backend routing:

1. **Time-sensitive guard** — detects temporal keywords and goes straight to **SerpApi**, bypassing the cache entirely.  
2. **FAISS lookup** — for stable questions, checks if a semantically similar question was already answered. If yes, returns the cached answer instantly.  
3. **Backend fallback** — on a cache miss, calls **SerpApi** for an answer, then stores it for future hits.  
4. **JSON persistence** — cache entries (questions, embeddings, answers) are saved to disk so the index survives notebook restarts.  
5. **Latency logging** — every call reports whether it was a hit, miss, or live search, and how long it took.

---

### What Should (and Should NOT) Be Semantically Cached?

The questions below are AWS concepts — stable by nature, so they're excellent cache candidates. The exceptions are anything that requires live, up-to-the-minute data.

#### ✅ Good to cache — stable AWS documentation answers:
| Question | Why it's safe to cache |
|---|---|
| *"What is an S3 bucket in AWS?"* | Core concept, always the same |
| *"How does AWS Lambda work?"* | Stable service behaviour |
| *"What is AWS IAM?"* | Conceptual definition from docs |
| *"What is the difference between EC2 and ECS?"* | Architectural comparison |
| *"How does Amazon CloudFront work?"* | Service explanation |
| *"What is an AWS VPC?"* | Networking concept |

#### ❌ Do NOT cache — time-sensitive, answers change even for AWS:
| Question | Why it must NOT be cached | Backend |
|---|---|---|
| *"Are there any AWS outages right now?"* | Status changes minute to minute | SerpApi |
| *"What are the latest AWS features this week?"* | New releases announced daily | SerpApi |
| *"What is the current EC2 pricing today?"* | AWS updates pricing periodically | SerpApi |
| *"Is AWS S3 down right now?"* | Real-time health check | SerpApi |
| *"What new services did AWS announce this month?"* | New info every month | SerpApi |

The `is_time_sensitive()` method catches these using a keyword list and routes them to SerpApi — they never touch the FAISS index.

In [7]:
from sentence_transformers import SentenceTransformer  # Load Nomic embed model (import before faiss — see above)
import faiss            # Efficient similarity search over vector embeddings
import json             # Read/write cache from a JSON file
import numpy as np      # Numerical operations on embeddings
from transformers import AutoTokenizer, AutoModelForCausalLM  # (Optional) LLM for answer gen
import time             # Measure latency

class SemanticCaching:
    """
    A semantic cache that routes queries to the right backend:

    ┌─────────────────────────────────────────────────────────┐
    │  Query                                                   │
    │    │                                                     │
    │    ├─ Time-sensitive? ──YES──▶ SerpApi (live search)     │
    │    │                           NOT cached                │
    │    │                                                     │
    │    └─ Stable? ──▶ FAISS lookup                          │
    │                     │                                    │
    │                     ├─ HIT  ──▶ return cached answer ⚡  │
    │                     │                                    │
    │                     └─ MISS ──▶ SerpApi (live search)    │
    │                                 store → return           │
    └─────────────────────────────────────────────────────────┘
    """

    # Keywords that signal the question is time-sensitive and must NOT be cached.
    # Answers to these questions change over time — caching would return stale results.
    TIME_SENSITIVE_KEYWORDS = [
        "today", "tonight", "now", "currently", "current",
        "latest", "recent", "recently", "right now", "at the moment",
        "at present", "as of now", "this week", "this month", "this year",
        "this quarter", "this season", "this morning", "this afternoon",
        "this evening", "this weekend", "yesterday", "tomorrow",
        "last week", "last month", "last year", "upcoming", "live",
        "breaking", "just happened", "what time", "what day", "what date",
        "happening now", "events today", "news today", "news this week",
        "stock price", "share price", "weather", "forecast", "temperature",
        "real-time", "realtime", "schedule today", "outage", "down right now",
        "is aws down", "aws status",
    ]

    def __init__(self, json_file='cache.json', clear_on_init=False):
        # Initialize Faiss index with Euclidean distance
        self.index = faiss.IndexFlatL2(768)
        if self.index.is_trained:
            print('Index trained')

        # Initialize Sentence Transformer model
        self.encoder = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

        # Euclidean distance threshold for cache hits (lower = stricter)
        self.euclidean_threshold = 0.2

        # JSON file to persist cache entries
        self.json_file = json_file

        # Load cache or clear already loaded cache
        if clear_on_init:
            self.clear_cache()
        else:
            self.load_cache()

    # ------------------------------------------------------------------
    # Time-sensitivity detection
    # ------------------------------------------------------------------

    def is_time_sensitive(self, question: str) -> bool:
        """
        Returns True if the question is time-sensitive and should NOT be cached.

        Time-sensitive questions reference current events, live data, or time-bound
        information whose answers change frequently. They still go to SerpApi — they
        just bypass the cache in both directions: never read, never written.

        Examples that return True (→ SerpApi, never cached):
            'Are there any AWS outages right now?'
            'What are the latest AWS features released this week?'
            'What is the current EC2 pricing today?'
            'Is AWS S3 down right now?'

        Examples that return False (→ check cache, then SerpApi on a miss):
            'What is an S3 bucket in AWS?'
            'How does AWS Lambda work?'
            'What is AWS IAM?'
            'What is the difference between EC2 and ECS?'
        """
        question_lower = question.lower()
        return any(keyword in question_lower for keyword in self.TIME_SENSITIVE_KEYWORDS)

    # ------------------------------------------------------------------
    # Cache persistence
    # ------------------------------------------------------------------

    def clear_cache(self):
        """Clears in-memory cache, resets FAISS index, and overwrites the JSON file."""
        self.cache = {
            'questions': [],
            'embeddings': [],
            'answers': [],
            'response_text': []
        }
        self.index = faiss.IndexFlatL2(768)
        self.save_cache()
        print("Semantic cache cleared.")

    def load_cache(self):
        """Load existing cache or initialize empty structure."""
        try:
            with open(self.json_file, 'r') as file:
                self.cache = json.load(file)
        except FileNotFoundError:
            self.cache = {'questions': [], 'embeddings': [], 'answers': [], 'response_text': []}

    def save_cache(self):
        """Persist cache back to disk."""
        with open(self.json_file, 'w') as file:
            json.dump(self.cache, file)

    # ------------------------------------------------------------------
    # Main query method
    # ------------------------------------------------------------------

    def ask(self, question: str) -> str:
        """
        Route the question to the correct backend and return an answer.

        Routing logic:
          1. Time-sensitive  → SerpApi — answer NOT cached
          2. Cache HIT       → return stored answer instantly, no backend call
          3. Cache MISS      → SerpApi — answer stored for next time
        """
        start_time = time.time()

        # ── 1. Time-sensitivity guard ─────────────────────────────────
        # Live search via SerpApi — result intentionally not stored
        if self.is_time_sensitive(question):
            print("⏰ Time-sensitive question — routing to SerpApi (live search, not cached).")
            response_text = search_web(question)
            print(f"Time taken: {time.time() - start_time:.3f}s")
            return response_text

        try:
            # ── 2. Cache lookup ───────────────────────────────────────
            embedding = self.encoder.encode([question], normalize_embeddings=True)
            D, I = self.index.search(embedding, 1)

            if D[0] >= 0:
                if I[0][0] != -1 and D[0][0] <= self.euclidean_threshold:
                    row_id = int(I[0][0])
                    print(f'✅ Cache hit at row: {row_id} | similarity: {1 - D[0][0]:.4f}')
                    print(f"Time taken: {time.time() - start_time:.3f}s")
                    return self.cache['response_text'][row_id]

            # ── 3. Cache miss → SerpApi ───────────────────────────────
            answer, response_text = self.generate_answer(question)

            self.cache['questions'].append(question)
            self.cache['embeddings'].append(embedding[0].tolist())
            self.cache['answers'].append(answer)
            self.cache['response_text'].append(response_text)
            self.index.add(embedding)
            self.save_cache()
            print(f"Time taken: {time.time() - start_time:.3f}s")

            return response_text

        except Exception as e:
            raise RuntimeError(f"Error during 'ask' method: {e}")

    def generate_answer(self, question: str):
        """
        Call the backend for a question the cache could not answer.

        Kept as its own method so the backend is swappable: point it at a document
        RAG pipeline, a vector store, or an LLM and the caching logic above is
        unchanged. That is what notebook 003 does.

        Returns:
            tuple: (raw backend result, answer string)
        """
        try:
            result = search_web(question)
            return result, result
        except Exception as e:
            raise RuntimeError(f"Error during 'generate_answer' method: {e}")

In [8]:
# Instantiate the semantic cache: builds/loads FAISS index, encoder, and JSON cache
# cache = SemanticCaching()

# Uncomment and use to re-instantiate the semantic cache and clear exisitng cache entries
cache = SemanticCaching(clear_on_init=True)

Index trained


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Semantic cache cleared.


### Testing the Semantic Cache

We validate the `SemanticCaching` class with stable AWS questions — conceptual, and
therefore safe to reuse an answer for.

Watch the routing in action:
- **First ask** of a question → cache miss → SerpApi → answer stored
- **Rephrased version** of the same question → cache hit → instant return, no API call
- **Time-sensitive question** → SerpApi every time → never stored

In [10]:
# Q1: Cache miss — SerpApi answers, result is stored
question1 = "What is an S3 bucket in AWS?"
answer1 = cache.ask(question1)
print(answer1)

# Q2: Cache miss — different AWS service
question2 = "How does AWS Lambda work?"
answer2 = cache.ask(question2)
print(answer2)

question3 = "What is Amazon Simple Queue Service (Amazon SQS)?"
answer3 = cache.ask(question3)
print(answer3)

question4 = "What is Amazon DynamoDB and when should I use it?"
answer4 = cache.ask(question4)
# Note:
# All are distinct enough to each get a separate backend call.
# Next, we'll test cache hits with rephrased versions of these questions.

Traversaal Pro: request successful.
Time taken: 1.483s
Unable to generate a response right now due to an LLM provider connection issue.
Traversaal Pro: request successful.
Time taken: 1.583s
Unable to generate a response right now due to an LLM provider connection issue.
Traversaal Pro: request successful.
Time taken: 1.055s
Unable to generate a response right now due to an LLM provider connection issue.
Traversaal Pro: request successful.
Time taken: 1.166s


In [11]:
# Cache HIT — rephrased version of Q1 ("What is an S3 bucket?")
# The FAISS index finds the stored embedding is similar enough → returns instantly
print(cache.ask("Can you explain what Amazon S3 buckets are?"))

Traversaal Pro: request successful.
Time taken: 1.028s
Unable to generate a response right now due to an LLM provider connection issue.


In [12]:
# Cache MISS — new question about DynamoDB → backend call + stored
print(cache.ask("What is Amazon DynamoDB and when should I use it?"))

✅ Cache hit at row: 3 | similarity: 1.0000
Time taken: 0.119s
Unable to generate a response right now due to an LLM provider connection issue.


In [13]:
# Cache MISS — different enough to not match DynamoDB → backend call
print(cache.ask("How does Amazon RDS differ from DynamoDB?"))

Traversaal Pro: request successful.
Time taken: 0.735s
Unable to generate a response right now due to an LLM provider connection issue.


### Testing the Time-Sensitivity Filter + Dual-Backend Routing

Here we demonstrate the full routing logic:

| Question type | Detected by | Backend | Cached? |
|---|---|---|---|
| Contains temporal keyword | `is_time_sensitive()` → `True` | **SerpApi** (live Google search) | ❌ Never |
| Stable AWS concept | `is_time_sensitive()` → `False` + cache miss | **SerpApi** | ✅ Stored |
| Previously seen question | `is_time_sensitive()` → `False` + cache hit | **FAISS cache** | ✅ Already stored |

**AWS-specific time-sensitive examples** — even though they're about AWS, these need live answers:
- *"Are there any AWS outages right now?"* → changes minute to minute  
- *"What are the latest AWS features released this week?"* → new announcements daily  
- *"What is the current EC2 pricing today?"* → pricing can be updated by AWS anytime  

**AWS stable examples** — conceptual, and they don't change:
- *"What is an S3 bucket?"* — always the same concept
- *"How does Lambda work?"* — core service behaviour is stable

In [14]:
# Classification check — see which questions are flagged before running any queries
time_sensitive_aws = [
    "Are there any AWS outages right now?",
    "What are the latest AWS features released this week?",
    "What is the current EC2 pricing today?",
    "Is AWS S3 down right now?",
    "What new services did AWS announce this month?",
    "What is the current AWS free tier limit as of now?",
]

stable_aws = [
    "What is an S3 bucket in AWS?",
    "How does AWS Lambda work?",
    "What is the difference between EC2 and ECS?",
    "What is an AWS VPC?",
]

print("=== Time-Sensitive AWS Questions (→ SerpApi, never cached) ===")
for q in time_sensitive_aws:
    flag = cache.is_time_sensitive(q)
    label = "⏰ SerpApi, never cached" if flag else "✅ SerpApi on miss, then cached"
    print(f"  [{label}] {q}")

print("\n=== Stable AWS Questions (→ SerpApi on miss, then cached) ===")
for q in stable_aws:
    flag = cache.is_time_sensitive(q)
    label = "⏰ SerpApi, never cached" if flag else "✅ SerpApi on miss, then cached"
    print(f"  [{label}] {q}")

=== Time-Sensitive AWS Questions (→ SerpApi, never cached) ===
  [⏰ SerpApi (live)] Are there any AWS outages right now?
  [⏰ SerpApi (live)] What are the latest AWS features released this week?
  [⏰ SerpApi (live)] What is the current EC2 pricing today?
  [⏰ SerpApi (live)] Is AWS S3 down right now?
  [⏰ SerpApi (live)] What new services did AWS announce this month?
  [⏰ SerpApi (live)] What is the current AWS free tier limit as of now?

=== Stable AWS Questions (→ Traversaal Pro on miss, cached) ===
  [✅ Traversaal Pro (cached)] What is an S3 bucket in AWS?
  [✅ Traversaal Pro (cached)] How does AWS Lambda work?
  [✅ Traversaal Pro (cached)] What is the difference between EC2 and ECS?
  [✅ Traversaal Pro (cached)] What is an AWS VPC?


In [15]:
# Time-sensitive AWS questions → routed to SerpApi, NEVER stored in cache
# Ask the same question twice — both calls go live, nothing accumulates in FAISS

print("--- Query A (time-sensitive: outage check) ---")
print(cache.ask("Are there any AWS outages right now?"))

print("\n--- Query A again (still time-sensitive → SerpApi again, not cached) ---")
print(cache.ask("Are there any AWS outages right now?"))

print("\n--- Query B (time-sensitive: pricing) ---")
print(cache.ask("What is the current EC2 pricing today?"))

# Verify the cache count has not grown due to these time-sensitive calls
print(f"\nCache entries (should be same as before): {len(cache.cache['questions'])}")
print("Cached questions:", cache.cache['questions'])

--- Query A (time-sensitive: outage check) ---
⏰ Time-sensitive question — routing to SerpApi (live search, not cached).
SerpApi: fetching live search results 🌐 ...
Time taken: 0.057s
[1] Service health - Sep 15, 2026 | AWS Health Dashboard | Global
    Mar 03 8:14 AM PST We are providing an update on the ongoing service disruptions affecting the AWS Middle East (UAE) Region (ME-CENTRAL-1). We continue to make ...
    Source: https://health.aws.amazon.com/

[2] AWS live status. Problems and outages for Amazon Web ...
    Amazon Web Services User reports show no current problems with Amazon Web Services
    Source: https://downdetector.com/status/aws-amazon-web-services/

[3] Service health - Sep 15, 2026 | AWS Health Dashboard | Global
    At this time, power has not yet been restored to the affected AZ. For now, we recommend continuing to retry any failed API requests.
    Source: https://health.aws.amazon.com/health/status?eventID=arn:aws:health:us-east-1::event/MULTIPLE_SERVICES/AWS

In [16]:
# Stable AWS question → cache miss on first call (SerpApi), cache hit on second
print("--- Query C (stable AWS, first call → SerpApi) ---")
print(cache.ask("How do you configure S3 bucket policies?"))

print("\n--- Query D (semantically similar to C → cache hit) ---")
print(cache.ask("What is the way to set up an S3 bucket access policy?"))

print(f"\nTotal cached entries now: {len(cache.cache['questions'])}")
print("Cached questions:", cache.cache['questions'])

--- Query C (stable AWS, first call → Traversaal Pro) ---
Traversaal Pro: request successful.
Time taken: 0.956s
Unable to generate a response right now due to an LLM provider connection issue.

--- Query D (semantically similar to C → cache hit) ---
✅ Cache hit at row: 6 | similarity: 0.8429
Time taken: 0.111s
Unable to generate a response right now due to an LLM provider connection issue.

Total cached entries now: 7
Cached questions: ['What is an S3 bucket in AWS?', 'How does AWS Lambda work?', 'What is Amazon Simple Queue Service (Amazon SQS)?', 'What is Amazon DynamoDB and when should I use it?', 'Can you explain what Amazon S3 buckets are?', 'How does Amazon RDS differ from DynamoDB?', 'How do you configure S3 bucket policies?']
